## Imports

In [1]:
from qsopt import * 
import numpy as np
import jax.numpy as jnp
import optax

## Define experimental parameters

In [2]:
# Define custom physical constants
custom_constants = PhysicalConstants(
    chi=0.75,                    # Dispersive coupling
    photon_cavity_coupling=1.5,  # Photon-cavity coupling
    inverse_pulse_width=0.2      # Inverse pulse width
)

# Define custom system dimensions
custom_dims = SystemDimensions(
    cavity_levels=2,
    qubit_levels=2,
    field_levels=2
)

# Define measurement protocol
custom_measurement = MeasurementProtocol(
    measurement_times = [-5.0, 0.0, 5.0]
)

# Define initial state configuration (SINGLE_PHOTON)
initial_state = InitialStateConfig(
    state_type=InitialStateType.SINGLE_PHOTON
)

# Define noise configuration
noise_config = NoiseConfiguration(
    depolarizing=0.01,  
    dephasing=0.005,      
    relaxation=0.01     
)

# Create parameters with custom configuration
exp_parameters = ExperimentalParameters(
    physical_constants=custom_constants,
    system_dims=custom_dims,
    measurement=custom_measurement,
    initial_state=initial_state,
    noise_config=noise_config
)

print(exp_parameters)


SYSTEM DIMENSIONS
------------------------------
  Cavity levels:             2
  Qubit levels:              2
  Field levels:              2
  Total dimension:           8
  Status:               VALID

PHYSICAL CONSTANTS
------------------------------
  Chi:                    0.7500
  Photon cavity coupling: 1.5000
  Inverse pulse width:    0.2000
  Status:               VALID

MEASUREMENT PROTOCOL
------------------------------
  Number of measurements:      3
  Measurement times: [-5.0, 0.0, 5.0]
  Status:               VALID

INITIAL STATE
------------------------------
  Type:                 single_photon

NOISE MODEL
------------------------------
  Depolarizing rate:      0.0100
  Dephasing rate:         0.0050
  Relaxation rate:        0.0100
  Total noise rate:       0.0250
  Custom operators:     None
  Status:               VALID

SYSTEM STATUS
------------------------------
  Configuration:        VALID


In [3]:
parameters = TrainableParameters()
parameters.add_rotation_angles(['ry1', 'ry2'], [1., 1.], optimizer=optax.adam(0.01))

print(parameters)

TrainableParameters(total=2, rotation_angles=2, measurement_times=0, custom=0)


In [4]:
experiment = SingleQubitExperiment(exp_parameters, parameters)


## Test Single Simulation

Before optimization, let's run a single simulation to see the initial state of the system.

In [ ]:
# Get initial state
rho0 = experiment.get_initial_state()
print(f"Initial state shape: {rho0.shape}")
print(f"Initial state trace: {rho0.tr():.6f}")

# Get solvers
solver_with = experiment.get_solver_with_interaction()
solver_without = experiment.get_solver_no_interaction()

# Get initial parameter values
theta1 = parameters.parameters[0].value
theta2 = parameters.parameters[1].value

print(f"\nInitial parameters:")
print(f"  θ₁ = {theta1:.4f} rad")
print(f"  θ₂ = {theta2:.4f} rad")

# Create measurement dictionary
measurements = {t: 0.0 for t in exp_parameters.measurement.measurement_times}

# Run simulation
prob_with = experiment.simulation(solver_with, rho0, theta1, theta2, measurements)
prob_without = experiment.simulation(solver_without, rho0, theta1, theta2, measurements)

initial_contrast = prob_with - prob_without

print(f"\nSimulation results:")
print(f"  P(with interaction):    {prob_with:.6f}")
print(f"  P(without interaction): {prob_without:.6f}")
print(f"  Contrast:               {initial_contrast:.6f}")

## Run Optimization

Now let's optimize the parameters to maximize the detection contrast.

In [ ]:
# Run optimization for 20 steps
print("Starting optimization...\n")

history = experiment.optimize(
    num_steps=20,
    learning_rate=0.05,
    verbose=True
)

print("\nOptimization complete!")

## View Results

In [ ]:
# Get final values
final_theta1 = parameters.parameters[0].value
final_theta2 = parameters.parameters[1].value
final_contrast = history['contrast'][-1]

print("="*60)
print("OPTIMIZATION SUMMARY")
print("="*60)
print(f"\nInitial Parameters:")
print(f"  θ₁ = {theta1:.4f} rad = {np.degrees(theta1):.2f}°")
print(f"  θ₂ = {theta2:.4f} rad = {np.degrees(theta2):.2f}°")
print(f"  Contrast = {initial_contrast:.6f}")

print(f"\nFinal Parameters:")
print(f"  θ₁ = {final_theta1:.4f} rad = {np.degrees(final_theta1):.2f}°")
print(f"  θ₂ = {final_theta2:.4f} rad = {np.degrees(final_theta2):.2f}°")
print(f"  Contrast = {final_contrast:.6f}")

